# Step 2 — PI-CAI Fold 0 Download and Data Exploration

**Goal:** Download PI-CAI Public Training fold 0 (~5.4 GB), unzip it, visualize the data, and prepare a 20-patient subset.

## Design Principle: Idempotent
Every cell is re-runnable. It checks state, skips if already done, completes if missing. Even after a Colab disconnect or kernel reset, you do not have to start from scratch.

## Execution Order
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells sequentially with **Shift+Enter**
3. On error: stop and inspect the output

## Expected Outputs
- `output/figures/sample_case_sequences.png` (presentation slide 4)
- `output/figures/sample_case_gt_overlays.png` (presentation slide 5)
- `output/metrics/subset20_inventory.csv` (20-patient list)
- `input/images/subset20/` (subset persisted to Drive)

## 0 — Mount Drive and define paths

In [ ]:
from google.colab import drive
if not __import__('os').path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

from pathlib import Path

# Persistent Drive paths
PROJECT_ROOT  = Path('/content/drive/MyDrive/Prostate_MRI_Project')
INPUT_DIR     = PROJECT_ROOT / 'input'
IMAGES_DIR    = INPUT_DIR / 'images'
LABELS_DIR    = INPUT_DIR / 'picai_labels'
OUTPUT_DIR    = PROJECT_ROOT / 'output'
FIGURES_DIR   = OUTPUT_DIR / 'figures'
METRICS_DIR   = OUTPUT_DIR / 'metrics'
DRIVE_SUBSET  = IMAGES_DIR / 'subset20'

# Ephemeral Colab local disk (fast)
LOCAL_WORK    = Path('/content/picai_work')
LOCAL_ZIP     = LOCAL_WORK / 'fold0.zip'
LOCAL_IMAGES  = LOCAL_WORK / 'images'

for d in [INPUT_DIR, IMAGES_DIR, LABELS_DIR, OUTPUT_DIR, FIGURES_DIR, METRICS_DIR,
          LOCAL_WORK, LOCAL_IMAGES, DRIVE_SUBSET]:
    d.mkdir(parents=True, exist_ok=True)

ZENODO_URL = 'https://zenodo.org/api/records/6624726/files/picai_public_images_fold0.zip/content'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'LOCAL_WORK:   {LOCAL_WORK}')

## 1 — Check and install required packages

If the kernel was reset or the VM reassigned, packages may be missing. This cell installs only what is missing.

In [ ]:
import importlib, sys, subprocess

required = {
    'SimpleITK':   'SimpleITK',
    'nibabel':     'nibabel',
    'matplotlib':  'matplotlib',
    'pandas':      'pandas',
    'monai':       'monai[all]==1.3.2',
    'requests':    'requests',
}

missing = []
for mod, pkg in required.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'Installing missing packages: {missing}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    print('\nInstallation complete. The pip warnings below are for Colab default packages we do not use.')
else:
    print('All packages already installed.')

import SimpleITK as sitk, nibabel as nib, matplotlib, pandas as pd, numpy as np, monai
print()
print(f'SimpleITK:  {sitk.Version_VersionString()}')
print(f'nibabel:    {nib.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'pandas:     {pd.__version__}')
print(f'numpy:      {np.__version__}')
print(f'MONAI:      {monai.__version__}')

## 2 — Disk usage

In [3]:
!df -h /content /content/drive/MyDrive 2>/dev/null

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   51G  186G  22% /
drive           236G   60G  177G  26% /content/drive


## 3 — Download fold 0 from Zenodo (if absent)

Idempotent: skips when the local zip already matches the expected size; uses `curl -C -` to resume if partial.

In [ ]:
import requests
import os

def check_zip_status():
    """Return one of: absent / partial / complete / oversized / unknown."""
    if not LOCAL_ZIP.exists():
        return 'absent', 0, 0
    local = LOCAL_ZIP.stat().st_size
    try:
        r = requests.head(ZENODO_URL, allow_redirects=True, timeout=30)
        expected = int(r.headers.get('content-length', 0))
    except Exception as e:
        print(f'Zenodo HEAD failed ({e}); cannot compare sizes.')
        return 'unknown', local, 0
    if local == expected:
        return 'complete', local, expected
    elif local < expected:
        return 'partial', local, expected
    else:
        return 'oversized', local, expected

status, local, expected = check_zip_status()
print(f'Local:    {local/1e9:.3f} GB ({local:,} bytes)')
print(f'Expected: {expected/1e9:.3f} GB ({expected:,} bytes)')
print(f'Status:   {status}\n')

if status == 'complete':
    print('Zip already complete; download skipped.')
elif status in ('absent', 'partial'):
    print('Starting download (5-20 min); curl -C - resumes from any partial file.')
    os.chdir(LOCAL_WORK)
    !curl -C - -L "{ZENODO_URL}" -o "{LOCAL_ZIP}"
    status, local, expected = check_zip_status()
    print(f'\nPost-download: {status} ({local/1e9:.3f} / {expected/1e9:.3f} GB)')
    if status != 'complete':
        raise RuntimeError(f'Download incomplete (status={status}). Re-run this cell.')
else:
    print(f'Unexpected status: {status}. Manual inspection required.')

## 4 — Unzip (if not yet extracted)

Idempotent: skips when content is already present on local disk.

In [ ]:
existing_mha = list(LOCAL_IMAGES.rglob('*.mha'))
existing_dirs = [d for d in LOCAL_IMAGES.iterdir() if d.is_dir()] if LOCAL_IMAGES.exists() else []

if len(existing_mha) > 100 or len(existing_dirs) > 50:
    print(f'Already extracted: {len(existing_dirs)} folders, {len(existing_mha)} .mha files. Unzip skipped.')
else:
    print('Unzipping (5-10 min)...')
    !unzip -q -o "{LOCAL_ZIP}" -d "{LOCAL_IMAGES}"
    existing_mha = list(LOCAL_IMAGES.rglob('*.mha'))
    existing_dirs = [d for d in LOCAL_IMAGES.iterdir() if d.is_dir()]
    print(f'\nUnzip complete: {len(existing_dirs)} folders, {len(existing_mha)} .mha files.')

## 5 — Patient inventory

Detect the folder layout and extract the patient ID list.

In [ ]:
import re

# PI-CAI has two possible layouts: (a) patient_folder/file.mha or (b) flat file.mha
patient_dirs = sorted([d for d in LOCAL_IMAGES.iterdir() if d.is_dir()])

if patient_dirs:
    structure = 'nested'
    print(f'Layout: nested (per-patient folders)')
    print(f'Number of patient folders: {len(patient_dirs)}')
    print(f'First 5: {[d.name for d in patient_dirs[:5]]}')
    sample_case_dir = patient_dirs[0]
    sample_files = sorted(sample_case_dir.glob('*.mha'))
    print(f'\nExample patient ({sample_case_dir.name}):')
    for f in sample_files:
        print(f'  {f.name:40s} {f.stat().st_size/1e6:6.1f} MB')
else:
    structure = 'flat'
    all_mha = sorted(LOCAL_IMAGES.rglob('*.mha'))
    print(f'Layout: flat')
    print(f'Number of .mha files: {len(all_mha)}')
    case_ids = sorted(set(re.match(r'(\d+_\d+)', p.name).group(1)
                          for p in all_mha if re.match(r'(\d+_\d+)', p.name)))
    print(f'Unique patient_id: {len(case_ids)}')
    print(f'First 5: {case_ids[:5]}')

print(f'\nstructure = {structure!r}')

## 6 — Sample patient: visualize T2W + ADC + HBV

For **slide 4**: show the three model-input sequences side by side.

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

def find_seq(files, suffix):
    """Find a file ending with _<suffix>.mha (e.g. _t2w.mha)."""
    for f in files:
        if f.name.lower().endswith(f'_{suffix}.mha'):
            return f
    return None

def load_mha(path):
    if path is None or not path.exists():
        return None, None
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)  # (Z, Y, X)
    return arr, img

if structure == 'nested':
    sample_case_id = patient_dirs[0].name
    sample_files = list(patient_dirs[0].glob('*.mha'))
else:
    sample_case_id = case_ids[0]
    sample_files = list(LOCAL_IMAGES.rglob(f'{sample_case_id}*.mha'))

t2w_path = find_seq(sample_files, 't2w')
adc_path = find_seq(sample_files, 'adc')
hbv_path = find_seq(sample_files, 'hbv')

print(f'Patient: {sample_case_id}')
print(f'  T2W: {t2w_path.name if t2w_path else "MISSING"}')
print(f'  ADC: {adc_path.name if adc_path else "MISSING"}')
print(f'  HBV: {hbv_path.name if hbv_path else "MISSING"}')

t2w, t2w_img = load_mha(t2w_path)
adc, _       = load_mha(adc_path)
hbv, _       = load_mha(hbv_path)

if t2w is not None:
    print(f'\nT2W shape: {t2w.shape}, spacing: {t2w_img.GetSpacing()}')
    print(f'T2W intensity range: [{t2w.min()}, {t2w.max()}]')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, vol) in zip(axes, [('T2W', t2w), ('ADC', adc), ('HBV (DWI)', hbv)]):
    if vol is None:
        ax.text(0.5, 0.5, f'{name}: missing', ha='center', va='center')
        ax.axis('off')
        continue
    mid = vol.shape[0] // 2
    ax.imshow(vol[mid], cmap='gray')
    ax.set_title(f'{name}  (slice {mid+1}/{vol.shape[0]})')
    ax.axis('off')

plt.suptitle(f'PI-CAI Multi-parametric MRI — Case {sample_case_id}', fontsize=14)
plt.tight_layout()

fig_path = FIGURES_DIR / 'sample_case_sequences.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

## 7 — Ground Truth overlay (slide 5)

Overlay the anatomy masks (whole gland, zonal) and the lesion mask (human_expert) on the T2W image for the same patient.

In [ ]:
import nibabel as nib

def find_label(pattern_dir, case_id):
    """Find a .nii.gz file whose name starts with `case_id`."""
    if not pattern_dir.exists():
        return None
    matches = list(pattern_dir.rglob(f'{case_id}*.nii.gz'))
    return matches[0] if matches else None

def load_nii(path):
    if path is None or not path.exists():
        return None
    # Convert nibabel's (X,Y,Z) layout to SITK's (Z,Y,X)
    return nib.load(str(path)).get_fdata().transpose(2, 1, 0)

lesion_path = find_label(LABELS_DIR / 'csPCa_lesion_delineations' / 'human_expert' / 'resampled', sample_case_id)
if lesion_path is None:
    lesion_path = find_label(LABELS_DIR / 'csPCa_lesion_delineations' / 'human_expert' / 'original', sample_case_id)
wg_path     = find_label(LABELS_DIR / 'anatomical_delineations' / 'whole_gland' / 'AI', sample_case_id)
zonal_path  = find_label(LABELS_DIR / 'anatomical_delineations' / 'zonal_pz_tz' / 'AI', sample_case_id)

print(f'Lesion (human):  {lesion_path}')
print(f'Whole gland AI:  {wg_path}')
print(f'Zonal AI:        {zonal_path}')

lesion_mask = load_nii(lesion_path)
wg_mask     = load_nii(wg_path)
zonal_mask  = load_nii(zonal_path)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
mid = t2w.shape[0] // 2

panels = [
    ('Whole Gland (AI)',              wg_mask,     'autumn'),
    ('Zonal PZ (=2) / TZ (=1) (AI)',  zonal_mask,  'viridis'),
    ('csPCa Lesion (Human)',          lesion_mask, 'Reds'),
]

for ax, (name, mask, cmap) in zip(axes, panels):
    ax.imshow(t2w[mid], cmap='gray')
    if mask is not None and mid < mask.shape[0]:
        masked = np.ma.masked_where(mask[mid] == 0, mask[mid])
        ax.imshow(masked, cmap=cmap, alpha=0.5)
        ax.set_title(f'{name}\n(slice {mid+1}/{t2w.shape[0]})')
    else:
        ax.set_title(f'{name}\n(no label for this patient)')
    ax.axis('off')

plt.suptitle(f'Ground Truth Overlays — Case {sample_case_id}', fontsize=14)
plt.tight_layout()
fig_path = FIGURES_DIR / 'sample_case_gt_overlays.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

## 8 — Marksheet and 20-patient subset selection

Select 10 csPCa-positive and 10 csPCa-negative patients, then persist the subset to Drive.

In [ ]:
import pandas as pd

marksheet_path = LABELS_DIR / 'clinical_information' / 'marksheet.csv'
marksheet = pd.read_csv(marksheet_path)
marksheet['patient_str'] = marksheet['patient_id'].astype(str)

print(f'Marksheet: {len(marksheet)} patients')
print(f'Columns: {list(marksheet.columns)}')
marksheet.head()

In [ ]:
if structure == 'nested':
    local_patient_ids = sorted(set(d.name.split('_')[0] for d in patient_dirs))
else:
    local_patient_ids = sorted(set(c.split('_')[0] for c in case_ids))

print(f'Total local patients: {len(local_patient_ids)}')

local_in_marksheet = marksheet[marksheet['patient_str'].isin(local_patient_ids)]
print(f'Matched in marksheet: {len(local_in_marksheet)}')

# Positive / negative split via the case_csPCa column (YES / NO)
pos = local_in_marksheet[local_in_marksheet['case_csPCa'] == 'YES'].head(10)
neg = local_in_marksheet[local_in_marksheet['case_csPCa'] == 'NO'].head(10)

print(f'\nPositive (csPCa YES) selected: {len(pos)}')
print(f'Negative (csPCa NO)  selected: {len(neg)}')
print(f'\nPositive IDs: {list(pos["patient_str"])}')
print(f'Negative IDs: {list(neg["patient_str"])}')

subset_ids = list(pos['patient_str']) + list(neg['patient_str'])

In [ ]:
import shutil

copied_new = 0
for pid in subset_ids:
    if structure == 'nested':
        matches = [d for d in patient_dirs if d.name.startswith(pid)]
        for src in matches:
            dst = DRIVE_SUBSET / src.name
            if not dst.exists():
                shutil.copytree(src, dst)
                copied_new += 1
    else:
        srcs = list(LOCAL_IMAGES.rglob(f'{pid}*.mha'))
        case_dir = DRIVE_SUBSET / pid
        case_dir.mkdir(exist_ok=True)
        for src in srcs:
            dst = case_dir / src.name
            if not dst.exists():
                shutil.copy(src, dst)
                copied_new += 1

print(f'Newly copied: {copied_new} files/folders')
print(f'\nDrive subset status:')
!du -sh "{DRIVE_SUBSET}"
!ls "{DRIVE_SUBSET}" | head -25

## 9 — Subset inventory CSV (for the presentation)

In [ ]:
subset_info = pd.concat([
    pos.assign(group='positive'),
    neg.assign(group='negative'),
], ignore_index=True)

show_cols = ['patient_id', 'study_id', 'group', 'case_csPCa', 'case_ISUP',
             'patient_age', 'psa', 'prostate_volume', 'center']
show_cols = [c for c in show_cols if c in subset_info.columns]
subset_summary = subset_info[show_cols]

out_csv = METRICS_DIR / 'subset20_inventory.csv'
subset_summary.to_csv(out_csv, index=False)
print(f'Inventory CSV saved: {out_csv}\n')
subset_summary

---

## Step 2 Complete

**Produced:**
- `output/figures/sample_case_sequences.png` — T2W/ADC/HBV (slide 4)
- `output/figures/sample_case_gt_overlays.png` — Ground truth overlays (slide 5)
- `output/metrics/subset20_inventory.csv` — 10 positive + 10 negative patient list
- `input/images/subset20/` — imaging data persisted to Drive

**Next:** `step3_anatomy_inference.ipynb` — zonal segmentation inference with the MONAI bundle.

---

### If the kernel resets or Colab disconnects
This notebook is idempotent. Just re-run from the top — every cell inspects state and either skips or completes. Download, unzip, and copy are all cached.